In [25]:
from sentence_transformers import SentenceTransformer

In [26]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3364.96it/s]


In [27]:
def is_heading(block):
    block = block.strip()
    words = block.split()

    return (
        len(words) <= 12
        and not block.endswith(".")
        and not block.endswith(";")
        and "\n" not in block
    )

In [28]:
import os

files = [
    "data/raw/egg.txt",
    "data/raw/milk.txt",
    "data/raw/storage.txt",
    "data/raw/fruits_and_vegetable.txt"
]

documents = []

for file in files:

    document_name = os.path.basename(file).replace(".txt", "")
    current_section = ""

    with open(file, "r", encoding="utf-8") as f:
        text = f.read()

    blocks = [
        b.strip()
        for b in text.split("\n\n")
        if b.strip()
    ]

    for block in blocks:

        if "\n" in block:
            first_line, remaining_text = block.split("\n", 1)

            if is_heading(first_line):
                current_section = first_line.strip()

                documents.append({
                    "document": document_name,
                    "section": current_section,
                    "text": remaining_text.strip(),
                    "source": file
                })

            else:
                documents.append({
                    "document": document_name,
                    "section": current_section,
                    "text": block.strip(),
                    "source": file
                })

        elif is_heading(block):
            current_section = block.strip()

        else:
            documents.append({
                "document": document_name,
                "section": current_section,
                "text": block.strip(),
                "source": file
            })

In [29]:
texts = [doc["text"] for doc in documents]

embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches: 100%|██████████| 4/4 [00:02<00:00,  1.34it/s]


In [30]:
embeddings.shape

(121, 384)

In [31]:
print(documents[0])

{'document': 'egg', 'section': 'Egg', 'text': 'egg, the content of the hard-shelled reproductive body produced by a bird, considered as food.\nWhile the primary role of the egg obviously is to reproduce the species, most eggs laid by domestic fowl, except those specifically set aside for hatching, are not fertilized but are sold mainly for human consumption. Eggs produced in quantity come from chickens, ducks, geese, turkeys, guinea fowl, pigeons, pheasants, and quail. This article describes the processing of chicken eggs, which represent the bulk of egg production in the United States and Europe. Duck eggs are consumed as food in parts of Europe and Asia, and goose eggs are also a food in many European countries. Commercial production of turkey and pigeon eggs is almost entirely confined to those used for producing turkey poults and young pigeons (squabs). Pheasant and quail eggs provide birds for hobby or sport use.', 'source': 'data/raw/egg.txt'}


In [32]:
import numpy as np
threshold = 0.75
max_words = 300
min_chunk_size = 150

chunks = []

current_chunk = {
    "id": 0,
    "title": documents[0]["section"],
    "text": documents[0]["text"],
    "source": documents[0]["source"]
}

for i in range(len(documents) - 1):

    e_i = embeddings[i]
    e_j = embeddings[i + 1]

    similarity = e_i.dot(e_j) / (
        np.linalg.norm(e_i) * np.linalg.norm(e_j)
    )

    current_word_count = len(current_chunk["text"].split())
    next_word_count = len(documents[i + 1]["text"].split())

    same_source = (
        current_chunk["source"] == documents[i + 1]["source"]
    )

    if (
        similarity >= threshold
        and same_source
        and current_word_count + next_word_count <= max_words
    ):

        current_chunk["text"] += "\n\n" + documents[i + 1]["text"]

    else:

        chunks.append(current_chunk)

        current_chunk = {
            "id": len(chunks),
            "title": documents[i + 1]["section"],
            "text": documents[i + 1]["text"],
            "source": documents[i + 1]["source"]
        }

# Save final chunk
chunks.append(current_chunk)


# -------------------------
# Second pass: merge small chunks
# -------------------------
merged_chunks = []

for chunk in chunks:

    word_count = len(chunk["text"].split())

    if merged_chunks:

        previous_word_count = len(merged_chunks[-1]["text"].split())

        if (
            word_count < min_chunk_size
            and merged_chunks[-1]["source"] == chunk["source"]
            and previous_word_count + word_count <= max_words
        ):

            merged_chunks[-1]["text"] += "\n\n" + chunk["text"]

        else:
            merged_chunks.append(chunk)

    else:
        merged_chunks.append(chunk)

chunks = merged_chunks


# -------------------------
# Reassign IDs
# -------------------------

for i, chunk in enumerate(chunks):
    chunk["id"] = i

In [33]:
len(chunks)

45

In [34]:
for i, chunk in enumerate(chunks):
    words = len(chunk["text"].split())
    print(f"Chunk {i}: {words} words")

Chunk 0: 237 words
Chunk 1: 130 words
Chunk 2: 218 words
Chunk 3: 283 words
Chunk 4: 237 words
Chunk 5: 230 words
Chunk 6: 106 words
Chunk 7: 160 words
Chunk 8: 237 words
Chunk 9: 247 words
Chunk 10: 290 words
Chunk 11: 227 words
Chunk 12: 248 words
Chunk 13: 99 words
Chunk 14: 197 words
Chunk 15: 212 words
Chunk 16: 283 words
Chunk 17: 22 words
Chunk 18: 173 words
Chunk 19: 289 words
Chunk 20: 223 words
Chunk 21: 256 words
Chunk 22: 238 words
Chunk 23: 292 words
Chunk 24: 216 words
Chunk 25: 213 words
Chunk 26: 217 words
Chunk 27: 161 words
Chunk 28: 271 words
Chunk 29: 160 words
Chunk 30: 190 words
Chunk 31: 284 words
Chunk 32: 167 words
Chunk 33: 292 words
Chunk 34: 288 words
Chunk 35: 263 words
Chunk 36: 170 words
Chunk 37: 252 words
Chunk 38: 300 words
Chunk 39: 288 words
Chunk 40: 222 words
Chunk 41: 241 words
Chunk 42: 251 words
Chunk 43: 62 words
Chunk 44: 189 words


In [35]:
for chunk in chunks:

    text = chunk["text"]

    embedding = model.encode(
        text,
        convert_to_numpy=True
    )

    chunk["embedding"] = embedding.tolist()

In [36]:
import json

with open("data/processed/chunks.json", "w", encoding="utf-8") as f:
    json.dump(
        chunks,
        f,
        indent=4,
        ensure_ascii=False
    )

In [37]:
print(type(chunks))
print(type(chunks[0]))
print(chunks[0].keys())

<class 'list'>
<class 'dict'>
dict_keys(['id', 'title', 'text', 'source', 'embedding'])
